In [0]:
%sql
create or replace table pricing_analytics.silver.productscdtype2_dim1
select distinct PRODUCT_NAME,
PRODUCTGROUP_NAME,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.daily_pricing where LAKE_HOUSE_UPDATED_DATE>(select coalesce(max(process_table_date_time), date('1900-01-01')) from pricing_analytics.bronze.pipeline_control_logs where process_name='pricing_data_silver_product_scdtype2' and process_status='Completed')

In [0]:
%sql
create or replace table pricing_analytics.silver.productscdtype2_dim2
select 
silverDim.PRODUCT_NAME,
silverDim.PRODUCTGROUP_NAME,
goldDim.PRODUCT_ID as GOLD_PRODUCT_ID,
goldDim.PRODUCT_NAME as GOLD_PRODUCT_NAME,
row_number() over(order by silverDim.PRODUCT_NAME, silverDim.PRODUCTGROUP_NAME) as PRODUCT_ID,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.productscdtype2_dim1 silverDim
left join pricing_analytics.gold.product_scdtype2_dim goldDim
on silverDim.PRODUCT_NAME=goldDim.PRODUCT_NAME
where goldDim.PRODUCT_NAME is null or silverDim.PRODUCTGROUP_NAME<>goldDim.PRODUCTGROUP_NAME

In [0]:
%sql
create or replace table pricing_analytics.silver.productscdtype2_dim3
select 
PRODUCT_NAME,
PRODUCTGROUP_NAME,
(PRODUCT_ID+PREV_MAX_ID) as PRODUCT_ID,
GOLD_PRODUCT_ID,
case when GOLD_PRODUCT_NAME is null then 'NEW' else 'CHENGED' end as RECORD_STATUS,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.productscdtype2_dim2
cross join(select coalesce(max(PRODUCT_ID), 0) as PREV_MAX_ID from pricing_analytics.gold.product_scdtype2_dim)

In [0]:
%sql
merge into pricing_analytics.gold.product_scdtype2_dim goldDim
using pricing_analytics.silver.productscdtype2_dim3 silverDim
on goldDim.PRODUCT_ID=silverDim.GOLD_PRODUCT_ID

when matched then update set
goldDim.RECORD_STATUS='CHENGED',
goldDim.effective_end_date=current_timestamp(),
goldDim.lake_house_updated_date=current_timestamp()

when not matched then insert
(PRODUCT_NAME, PRODUCTGROUP_NAME, PRODUCT_ID, RECORD_STATUS, effective_start_date, effective_end_date, lake_house_inserted_date, lake_house_updated_date) values 
(PRODUCT_NAME, PRODUCTGROUP_NAME, PRODUCT_ID, 'NEW', current_timestamp(), Null, current_timestamp(), current_timestamp())

In [0]:
%sql
insert into pricing_analytics.gold.product_scdtype2_dim(
  select PRODUCT_NAME,
  PRODUCTGROUP_NAME,
  PRODUCT_ID,
  'NEW',
  current_timestamp(),
  Null,
  current_timestamp(),
  current_timestamp()
  from pricing_analytics.silver.productscdtype2_dim3
  where RECORD_STATUS='CHENGED'
)

In [0]:
%sql
insert into pricing_analytics.bronze.pipeline_control_logs(
  select 
  'pricing_data_silver_product_scdtype2',
  max(LAKE_HOUSE_UPDATED_DATE),
  'Completed',
  current_timestamp()
  from pricing_analytics.silver.daily_pricing
)